In [ ]:
import numpy as np
import cv2 
%matplotlib inline
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

In [ ]:
import torch
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms,models
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [ ]:
main_path = "/kaggle/input/pets-facial-expression-dataset/Master Folder/"
train_path = main_path+"train/"
val_path = main_path+"valid/"
img_folder = ImageFolder(root=train_path)


# Class Counts and Different Image Sizes

In [ ]:
class_counts = {}
image_sizes = {}
for img, img_cls in img_folder:
    if img.size not in image_sizes:
        image_sizes[img.size] = 0
    image_sizes[img.size] += 1
    
    if img_cls not in class_counts:
        class_counts[img_cls] = 0

    class_counts[img_cls] += 1

print(class_counts)
print(image_sizes)

# Augmentations & Transforms

In [ ]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

data_transforms = {
    'train':
    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.01),
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize
    ]),
    'validation':
    transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        normalize
    ]),
}

# Dataset & Dataloader

In [ ]:
class PetDataset(Dataset):
    def __init__(self,root_dir,transform=None):
        self.dataset = ImageFolder(root = root_dir,transform=transform)
        self.transform = transform
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self,index):
        image , label  = self.dataset[index]
        samples = {'image':image,'label':label}
        return samples
        

In [ ]:
train_dataset = PetDataset(train_path,data_transforms['train'])
val_dataset = PetDataset(val_path,data_transforms['validation'])

In [ ]:
train_loader = DataLoader(train_dataset,batch_size = 16,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size = 16,shuffle=False)

In [ ]:
# Display image and label.
batch = next(iter(train_loader))
print(f"Feature batch shape: {batch['image'].size()}")
print(f"Labels batch shape: {batch['label'].size()}")
img = batch['image'][3].squeeze()
label = batch['label'][3]
plt.imshow(img.permute(1, 2, 0))
plt.show()
print(f"Label: {label}")

# Model

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

In [ ]:
class block(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        """
        A single Residual Block in the ResNet model.
        Args:
            in_channels: Number of input channels.
            out_channels: Number of output channels.
            stride: Stride for the convolutional layer.
            downsample: Downsampling layer to match dimensions for the skip connection.
        """
        super(block, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        """
        Forward pass through the residual block.
        Args:
            x: Input tensor.
        Returns:
            Tensor after passing through the residual block.
        """
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out



class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=4, input_size=(3, 224, 224)):
        """
        ResNet model using Residual Blocks.
        Args:
            block: Residual block class.
            layers: List defining the number of blocks in each layer.
            num_classes: Number of output classes for classification.
            input_size: Tuple representing the input size (C, H, W).
        """
        super(ResNet, self).__init__()
        self.in_channels = 64

        # Initial convolutional layer
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # Dynamically calculate the flattened size
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully connected layer
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride=1):
        """
        Creates a ResNet layer with the specified number of blocks.
        Args:
            block: Residual block class.
            out_channels: Number of output channels.
            blocks: Number of blocks in the layer.
            stride: Stride for the first block.
        """
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

        layers = [block(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def _get_flattened_size(self, input_size):
        """
        Calculate the size of the flattened tensor after passing through the feature extractor.
        Args:
            input_size: Tuple representing the input size (C, H, W).
        Returns:
            int: Flattened size of the tensor.
        """
        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_size)  # Batch size 1
            features = self._forward_features(dummy_input)
            return features.numel()

    def _forward_features(self, x):
        """
        Forward pass through the feature extractor (without the fully connected layer).
        """
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = self.avgpool(x).view(x.shape[0], -1) # (N, C, H_avg ,W_avg) -> (N, C*H_avg*W_avg)
        x = self.fc(x)
        return x



def resnet18(input_size=(3, 224, 224), num_classes=4):
    """
    Constructs a ResNet-18 model.
    Args:
        input_size: Tuple representing the input size (C, H, W).
        num_classes: Number of output classes.
    """
    return ResNet(block, [2, 2, 2, 2], num_classes=num_classes, input_size=input_size)


In [ ]:
model = resnet18()
model.to(device)

# Sample Batch

In [ ]:
batch = next(iter(train_loader))
model(batch['image'].to(device)).shape

In [ ]:
test_output = model(batch['image'].to(device))
print(test_output.argmax(dim=1))

# Loss and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training Val Loop

In [ ]:
def train_loop(model,loader,epoch):
    train_pred = []
    train_labels = []
    train_loss = 0
    total_samples = 0
    for i,data in enumerate(loader):
        optimizer.zero_grad()
        image = data['image'].to(device)
        label = data['label'].to(device)
        
        outputs = model(image)
        loss = criterion(outputs,label)
        
        loss.backward()
        optimizer.step()
        
        train_loss+= loss.item()
        train_pred.append(torch.argmax(outputs,dim = 1).cpu().detach().numpy())
        train_labels.append(label.cpu().detach().numpy())
        total_samples += data['image'].size(0) 
        
    print("Train Epoch: ",epoch,"| Loss: ",(train_loss/total_samples))
        
    return np.concatenate(train_pred).ravel(), np.concatenate(train_labels).ravel(),(train_loss/total_samples)


def val_loop(model,loader,epoch):
    val_pred = []
    val_labels = []
    val_loss = 0
    total_samples = 0
    with torch.no_grad():
        for i,data in enumerate(loader):
            optimizer.zero_grad()
            image = data['image'].to(device)
            label = data['label'].to(device)
            outputs = model(image)
            loss = criterion(outputs,label)
            val_loss+= loss.item()
            val_pred.append(torch.argmax(outputs,dim = 1).cpu().detach().numpy())
            val_labels.append(label.cpu().detach().numpy())
            total_samples += data['image'].size(0) 
            
        print("Val Epoch: ",epoch,"| Loss: ",(val_loss/total_samples))
        
        return np.concatenate(val_pred).ravel(), np.concatenate(val_labels).ravel(),(val_loss/total_samples)
        
        

In [ ]:
# import torch.optim.lr_scheduler as lr_scheduler


# scheduler = lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)  # Reduce LR every 1 epochs by a factor of 0.5

# num_epochs = 10
# train_preds = []
# train_labels = []
# train_loss_list = []

# val_preds = []
# val_labels = []
# val_loss_list = []

# for i in range(num_epochs):
#     current_lr = optimizer.param_groups[0]['lr']
#     print("Epoch: ", i, "| Learning Rate : ", current_lr)
    
#     model.train()
#     train_preds, train_labels, train_loss = train_val(model, train_loader, i + 1, "train")
#     train_loss_list.append(train_loss)
#     print("Train Accuracy: ", accuracy_score(train_labels, train_preds))
    
#     model.eval()
#     val_preds, val_labels, val_loss = train_val(model, val_loader, i + 1, "val")
#     val_loss_list.append(val_loss)
#     print("Val Accuracy: ", accuracy_score(val_labels, val_preds))
    
#     # Update the learning rate at the end of each epoch
#     scheduler.step()


In [ ]:
import os

# Directory to save models
save_dir = "/kaggle/working/saved_models"
os.makedirs(save_dir, exist_ok=True)

num_epochs = 100
patience = 10  # Number of epochs to wait before reducing the learning rate
learning_rate_decay_facor = 0.5
best_val_accuracy = 0
epochs_since_improvement = 0

train_preds = []
train_labels = []
train_loss_list = []

val_preds = []
val_labels = []
val_loss_list = []

for i in range(num_epochs):
    current_lr = optimizer.param_groups[0]['lr']
    print("Epoch: ", i, "| Learning Rate : ", current_lr)
    
    # Train phase
    model.train()
    train_preds, train_labels, train_loss = train_loop(model, train_loader, i + 1)
    train_loss_list.append(train_loss)
    print("Train Accuracy: ", accuracy_score(train_labels, train_preds))
    
    # Validation phase
    model.eval()
    val_preds, val_labels, val_loss = val_loop(model, val_loader, i + 1)
    val_loss_list.append(val_loss)
    val_accuracy = accuracy_score(val_labels, val_preds)
    print("Val Accuracy: ", val_accuracy)
    
    # Check for improvement
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_since_improvement = 0
        
        # Save the model
        save_path = os.path.join(save_dir, f"best_model_epoch_{i + 1}.pth")
        torch.save(model.state_dict(), save_path)
        print(f"New best model saved to {save_path}.")
    else:
        epochs_since_improvement += 1

    # Reduce the learning rate if no improvement for 'patience' epochs
    if epochs_since_improvement >= patience:
        current_lr = current_lr * learning_rate_decay_facor
        for param_group in optimizer.param_groups:
            param_group['lr'] = current_lr
        print(f"No improvement for {patience} epochs. Reducing learning rate to {current_lr}.")
        epochs_since_improvement = 0


In [ ]:
plt.plot([i+1 for i in range(num_epochs)],train_loss_list,'g',label='Training Loss')
plt.plot([i+1 for i in range(num_epochs)],val_loss_list,'r',label='Val Loss')
plt.title('Training and Testing loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
total_samples = 0

# # Iterate through the DataLoader and count samples
for batch in train_loader:
    total_samples += batch['image'].size(0)  # batch[0] contains the input data (images)

print("Sample train = ",total_samples)
total_samples = 0
for batch in val_loader:
    total_samples += batch['image'].size(0)  # batch[0] contains the input data (images)
print("Sample val = ", total_samples)